In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()
dim_machine = spark.read.table(f"{GOLD}.dim_machine").filter(col("__CURRENT") == True)
telemetry = spark.read.table(f"{SILVER}.telemetry_clean")

# 1. FACT UTILIZATION (Sessionization)
fact_util = telemetry.withColumn("is_active", when((col("rpm") > 500) & (col("engine_temp") > 40), 1).otherwise(0)) \
    .withColumn("date", to_date("sensor_ts")).groupBy("machine_id", "date").agg(sum("is_active").alias("active"), count("*").alias("total")) \
    .withColumn("active_hours", round((col("active") * 30.0) / 3600.0, 2)) \
    .withColumn("idle_hours", round(((col("total") - col("active")) * 30.0) / 3600.0, 2)) \
    .withColumn("utilization_pct", round(col("active") / col("total"), 2)) \
    .join(dim_machine.select("machine_sk", "machine_id"), "machine_id", "left") \
    .select("machine_sk", date_format("date", "yyyyMMdd").cast("int").alias("date_sk"), "active_hours", "idle_hours", "utilization_pct")
fact_util.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.fact_utilization_daily")

# 2. FACT MAINTENANCE
maintenance = spark.read.table(f"{SILVER}.maintenance_record")

fact_maint = maintenance.join(dim_machine.select("machine_sk", "machine_id"), "machine_id", "left") \
    .select(expr("xxhash64(maint_id)").alias("maint_sk"), "machine_sk", 
            date_format("completed_ts", "yyyyMMdd").cast("int").alias("date_sk"),
            "maint_type", col("cost").alias("maint_cost"), 
            datediff(current_date(), to_date("completed_ts")).alias("days_since_last_maintenance"))

fact_maint.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.fact_maintenance_event")

print("✅ gold_05_fact_ops built successfully!")